# Dataset curation

Sort, filter, group, dedupe, and flag structures to curate better training subsets.
Decisions (keep / remove + reason) persist to `data/cache/curation_decisions.json` and
survive across sessions. Everything runs on the cached subset built by
`python -m blockgen.data.build_cache`.

Workflow: **inspect → group → flag → save → apply**.


## Load

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from blockgen.curation import Curator

cur = Curator.from_cache(max_dim=24)
cur.load_decisions()      # resume any earlier session
cur.summary()

## Inspect — sort & filter by any feature
Features per structure: `sx,sy,sz, max_dim, n_blocks, density, footprint, height, n_block_types, dominant_block, dominant_frac, n_components, largest_component_frac`.

In [ ]:
# Biggest / densest first
cur.table(sort_by="n_blocks", limit=15)

In [ ]:
# Filtering returns a *view* (cur.<...>.indices maps back to the originals).
buildings = cur.filter(min_blocks=60, max_components=2, min_block_types=4)
tiny      = cur.filter(max_blocks=12)
monotype  = cur.filter(max_block_types=1)
print("building-like:", len(buildings), "| tiny:", len(tiny), "| single-material:", len(monotype))
buildings.table(sort_by="density", limit=8)

## Visual review — contact sheets
Render a grid of thumbnails for any set of indices.

In [ ]:
fig = cur.contact_sheet(buildings.indices[:12], cols=4); plt.show()

## Group similar structures
Two ways: **shape similarity** (occupancy IoU, GPU-accelerated) and **feature clusters** (k-means).

In [ ]:
# Near-duplicates (high IoU) — prime removal candidates.
dups = cur.find_duplicates(iou_threshold=0.95)
print(f"{len(dups)} near-duplicate groups; extras to drop = {sum(len(g)-1 for g in dups)}")
if dups:
    cur.show_group(dups[0], cols=len(dups[0])); plt.show()   # inspect the largest group

In [ ]:
# Looser shape families (tune the threshold).
families = [g for g in cur.group_by_similarity(iou_threshold=0.6) if len(g) > 2]
print("shape families (size>2):", len(families), "| sizes:", sorted((len(g) for g in families), reverse=True)[:10])

In [ ]:
# Feature clusters — coarse groupings by size/density/material composition.
clusters = cur.cluster_features(k=8)
for c, group in enumerate(clusters):
    rep = cur.features[group[0]]
    print(f"cluster {c}: n={len(group):4d}  e.g. {rep['n_blocks']} blocks, "
          f"density {rep['density']}, dominant {rep['dominant_block'].split(':')[-1]}")
# Visualize one cluster:
cur.show_group(clusters[0][:12], cols=4); plt.show()

## Flag for removal / keeping

`mark_remove(indices, reason)` and `mark_keep(indices, reason)` record decisions.
Pass the `.indices` of any filtered view, or members of a group.


In [ ]:
# Example pass — adjust to taste, then re-run:
if dups:
    cur.mark_remove([i for g in dups for i in g[1:]], reason="near-duplicate")  # keep 1 per group
cur.mark_remove(tiny.indices, reason="too small")
cur.mark_remove(cur.filter(predicate=lambda r: r["n_components"] >= 5 and r["largest_component_frac"] < 0.4).indices,
                reason="fragmented")
print("flagged remove:", len(cur.remove_list()))

## Save & apply
Persist decisions, then materialize the curated set for training.

In [ ]:
cur.save_decisions()                 # -> data/cache/curation_decisions.json
kept = cur.apply()                   # structures minus 'remove' (or only 'keep' if any keeps set)
print(f"curated set: {len(kept)} of {len(cur)} structures")

# `kept` is a list[Structure] ready to feed the training functions, e.g.:
#   from blockgen.utils.serialize import build_block_vocab
#   vocab = build_block_vocab(kept, max_dim=24)
#   from blockgen.training.train_ar import train_ar; train_ar(kept, vocab)
# Or export a path list for dataset_from_list_file:
# cur.export_keep_list("data/cache/curated_paths.txt")

# Metadata-powered curation (labeled cache)

The `data/raw` files don't map to any metadata, but the dataset's
`schematics/*.tfrecords` pair every schematic with its **source url** → rich
metadata (title, category, tags, description, popularity). Build the labeled
cache once, then curate with all of that available.

    python -m blockgen.data.tfrecord_dataset --max-dim 24   # one-time, ~minutes


In [ ]:
import os
from blockgen.data.tfrecord_dataset import TFCacheConfig, build_labeled_cache, cache_path
cfg = TFCacheConfig(max_dim=24)
if not os.path.exists(cache_path(cfg)):
    print(build_labeled_cache(cfg))

lab = Curator.from_labeled_cache(max_dim=24)
lab.load_decisions()
lab.summary()                 # now shows category + metadata coverage
print(lab.categories())

## Isolate an object type by metadata
Combine free-text `search` with category/feature filters to carve out one kind of build (e.g. houses).

In [ ]:
# Free-text search hits anything with 'house' in title/description/tags...
print("search('house'):", len(lab.search("house")))
# ...but that includes pixel-art/map-art. Constrain to buildable structures:
houses = (lab.search("house")
             .filter(category_in=["Land Structure Map", "Air Structure Map",
                                   "Other Map", "Complex Map"])
             .filter(min_blocks=60, max_components=3, min_block_types=3))
print("buildable houses:", len(houses))
fig = lab.contact_sheet(houses.indices[:12], cols=4); plt.show()

## Keep material/color variations, drop only true copies

Many "duplicates" are the *same build in different woods/wools* — valuable
variety, not junk. `find_variant_groups` surfaces them; `dedupe_keep_variants`
removes only exact (same-shape **and** same-palette) copies.


In [ ]:
exact = lab.find_exact_duplicates(0.95)
variants = lab.find_variant_groups(0.9)
print(f"exact-dup groups: {len(exact)} (extras {sum(len(g)-1 for g in exact)})  |  variant groups (KEEP): {len(variants)}")
if variants:
    lab.show_group(variants[0], cols=len(variants[0])); plt.show()   # same shape, different materials
n = lab.dedupe_keep_variants(reason="exact-duplicate")
print("flagged true-duplicate extras for removal:", n)

## Auto-mark a reliable seed subset by popularity
Highly-diamonded / downloaded builds are usually clean and complete — a good starting training set.

In [ ]:
n = lab.auto_mark_reliable(min_diamonds=10, min_downloads=100)
print("auto-marked 'keep' (popular):", n)
lab.save_decisions()

## Build & export a named subset for training
Materialize a single-category subset (e.g. houses) and hand it straight to the models.

In [ ]:
houses.export_keep_list("data/cache/subset_houses.txt")
house_structs = Curator.from_structures([houses.structures[i] for i in houses.indices],
                                         metadata=lab.metadata).apply()
print(f"house subset: {len(house_structs)} structures -> data/cache/subset_houses.txt")
# e.g.:  from blockgen.training.train_ar import train_ar
#        from blockgen.utils.serialize import build_block_vocab
#        v = build_block_vocab(house_structs, max_dim=24); train_ar(house_structs, v)

## Notes
- Decisions are keyed by source path (here the url), so they're stable across reloads.
- `cur.clear_marks(indices)` undoes flags; re-run `summary()` to see counts.
- Quick terminal pass: `python -m blockgen.curation.curate --labeled` (or omit `--labeled` for the raw cache).
